# 🏥 BrandHealth AI Pipeline — Google Colab Runner
Notebook này được thiết kế để tự động hóa toàn bộ quy trình chạy dự án **Phân loại sắc thái cảm xúc (Sentiment Analysis) tiếng Việt** sử dụng các mô hình học sâu (BiLSTM, Attention, PhoBERT) trên Google Colab.

### Quy trình bao gồm:
1. Kết nối Google Drive và Clone dự án.
2. Cài đặt các thư viện cần thiết.
3. Tải bộ dữ liệu NTC-SCV từ Hugging Face và thực hiện tiền xử lý tách từ tiếng Việt.
4. Huấn luyện các mô hình (chạy trên GPU, mỗi mô hình 5 lần → avg ± std).
5. Đánh giá kiểm thử mô hình trên tập Test.
6. Khởi chạy **Streamlit Web Dashboard** trực tiếp trên Colab thông qua kênh Tunnel.
7. Đẩy kết quả huấn luyện (logs, biểu đồ) ngược lại GitHub.

## Bước 1: Kết nối Google Drive & Clone Repository

In [19]:
!pwd

/content/PROJECT-NLP


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Clone dự án nếu chưa tồn tại thư mục
if not os.path.exists("/content/PROJECT-NLP"):
    !git clone https://github.com/kizzzbk/PROJECT-NLP.git /content/PROJECT-NLP

# Di chuyển vào thư mục dự án
%cd /content/PROJECT-NLP
# !git pull

Cloning into '/content/PROJECT-NLP'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 197 (delta 81), reused 177 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (197/197), 465.77 KiB | 2.35 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/PROJECT-NLP


## Bước 2: Cài đặt thư viện dependencies

In [3]:
print("Đang cài đặt các thư viện cần thiết...")
!pip install -r requirements.txt
!pip install datasets pyarrow wordcloud
print("\n=== Cài đặt thư viện thành công! ===")

Đang cài đặt các thư viện cần thiết...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 81.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 144.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 143.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 111.6 MB/s eta 0:00:00

=== Cài đặt thư viện thành công! ===


## Bước 3: Tải và tiền xử lý dữ liệu

In [4]:
print("--- 1. Tải dữ liệu NTC-SCV từ Hugging Face ---")
!python scripts/download_ntc_scv.py

print("\n--- 2. Tiền xử lý, tách từ ghép tiếng Việt & Chia tập dữ liệu ---")
# Thiết lập encoding UTF-8 để chạy mượt mà trên môi trường dòng lệnh Linux
os.environ['PYTHONIOENCODING'] = 'utf-8'
!python scripts/prepare_data.py

print("\n--- 3. Trực quan hóa thống kê dữ liệu ---")
!python scripts/visualize_dataset.py


--- 1. Tải dữ liệu NTC-SCV từ Hugging Face ---
Starting NTC-SCV dataset download from Hugging Face (thainq107/ntc-scv)...
README.md: 100% 570/570 [00:00<00:00, 2.28MB/s]
data/train-00000-of-00001.parquet: 100% 18.8M/18.8M [00:01<00:00, 12.3MB/s]
data/valid-00000-of-00001.parquet: 100% 6.35M/6.35M [00:00<00:00, 9.59MB/s]
data/test-00000-of-00001.parquet: 100% 6.35M/6.35M [00:00<00:00, 14.2MB/s]
Generating train split: 100% 30000/30000 [00:00<00:00, 138674.39 examples/s]
Generating valid split: 100% 10000/10000 [00:00<00:00, 152592.84 examples/s]
Generating test split: 100% 10000/10000 [00:00<00:00, 160770.91 examples/s]
Converting Hugging Face dataset splits to Pandas DataFrames...
  Loaded split 'train' with 30000 rows.
  Loaded split 'valid' with 10000 rows.
  Loaded split 'test' with 10000 rows.
Combined all splits into a single DataFrame with 50000 rows.
Using column 'sentence' as text and 'label' as label.

[SUCCESS] Successfully downloaded and saved NTC-SCV raw dataset to: /conten

## Bước 4: Huấn luyện các mô hình (Training)
Mỗi mô hình sẽ được huấn luyện **5 lần** (FR-2.2) với các seed khác nhau.
Kết quả cuối cùng: **avg ± std** của Accuracy và F1-score.

In [5]:
print("--- Huấn luyện mô hình BiLSTM (Baseline) — 5 lần ---")
!python scripts/train.py --model bilstm --num_runs 1

--- Huấn luyện mô hình BiLSTM (Baseline) — 5 lần ---
[16:17:33] INFO     Using GPU: Tesla T4 (14.6 GB)                   ]8;id=234053;file:///content/PROJECT-NLP/src/utils/device.py\device.py]8;;\:]8;id=146316;file:///content/PROJECT-NLP/src/utils/device.py#36\36]8;;\
[16:17:33] INFO     Loaded train: 35000 samples                      ]8;id=91161;file:///content/PROJECT-NLP/scripts/train.py\train.py]8;;\:]8;id=619176;file:///content/PROJECT-NLP/scripts/train.py#50\50]8;;\
           INFO     Loaded val: 5000 samples                         ]8;id=229258;file:///content/PROJECT-NLP/scripts/train.py\train.py]8;;\:]8;id=243962;file:///content/PROJECT-NLP/scripts/train.py#50\50]8;;\
           INFO     Loaded test: 10000 samples                       ]8;id=208496;file:///content/PROJECT-NLP/scripts/train.py\train.py]8;;\:]8;id=750800;file:///content/PROJECT-NLP/scripts/train.py#50\50]8;;\
           INFO                                                     ]

In [ ]:
# # 1. Điền thông tin GitHub của bạn vào đây
# GITHUB_TOKEN = ""  # Dán Token bạn vừa copy ở Bước 1 vào đây
# GITHUB_USER = "kizzzbk"
# REPO_NAME = "PROJECT-NLP"
# EMAIL = "phananh7105@gmail.com"

# # 2. Cấu hình định danh Git
# !git config --global user.email "{EMAIL}"
# !git config --global user.name "{GITHUB_USER}"

# # 3. Cập nhật lại Remote Origin chứa Token xác thực
# # Định dạng URL mới: https://<TOKEN>@github.com/<USER>/<REPO>.git
# authenticated_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
# !git remote set-url origin {authenticated_url}

# # 4. Thực hiện Push kết quả
# print("Đang tiến hành push lên GitHub...")
# !git add .
# !git commit -m "Upload training experiments and logs from Google Colab [skip ci]"
# !git push origin main
# print("=== Push thành công! ===")


Đang tiến hành push lên GitHub...
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
=== Push thành công! ===


In [6]:
print("--- Huấn luyện mô hình BiLSTM + Attention — 5 lần ---")
!python scripts/train.py --model bilstm_attention --num_runs 1

--- Huấn luyện mô hình BiLSTM + Attention — 5 lần ---
[16:36:49] INFO     Using GPU: Tesla T4 (14.6 GB)                   ]8;id=234053;file:///content/PROJECT-NLP/src/utils/device.py\device.py]8;;\:]8;id=146316;file:///content/PROJECT-NLP/src/utils/device.py#36\36]8;;\
[16:36:50] INFO     Loaded train: 35000 samples                      ]8;id=91161;file:///content/PROJECT-NLP/scripts/train.py\train.py]8;;\:]8;id=619176;file:///content/PROJECT-NLP/scripts/train.py#50\50]8;;\
           INFO     Loaded val: 5000 samples                         ]8;id=229258;file:///content/PROJECT-NLP/scripts/train.py\train.py]8;;\:]8;id=243962;file:///content/PROJECT-NLP/scripts/train.py#50\50]8;;\
           INFO     Loaded test: 10000 samples                       ]8;id=208496;file:///content/PROJECT-NLP/scripts/train.py\train.py]8;;\:]8;id=750800;file:///content/PROJECT-NLP/scripts/train.py#50\50]8;;\
           INFO                                                     

In [17]:
print("Đang tiến hành push lên GitHub...")
!git add .
!git commit -m "Upload training experiments and logs from Google Colab [skip ci]"
!git push origin main
print("=== Push thành công! ===")

Đang tiến hành push lên GitHub...
[main 0c880db] Upload training experiments and logs from Google Colab [skip ci]
 26 files changed, 1184 insertions(+)
 create mode 100644 experiments/2026-06-13_11-25-53_bilstm_attention_run_r1/config.yaml
 create mode 100644 experiments/2026-06-13_11-25-53_bilstm_attention_run_r1/hyperparams.json
 create mode 100644 experiments/2026-06-13_11-25-53_bilstm_attention_run_r1/metrics.json
 create mode 100644 experiments/2026-06-13_11-25-53_bilstm_attention_run_r1/summary.json
 create mode 100644 experiments/2026-06-13_11-25-53_bilstm_attention_run_r1/system_info.json
 create mode 100644 experiments/2026-06-13_11-32-44_bilstm_attention_run_r2/config.yaml
 create mode 100644 experiments/2026-06-13_11-32-44_bilstm_attention_run_r2/hyperparams.json
 create mode 100644 experiments/2026-06-13_11-32-44_bilstm_attention_run_r2/metrics.json
 create mode 100644 experiments/2026-06-13_11-32-44_bilstm_attention_run_r2/summary.json
 create mode 100644 experiments/2026-

In [ ]:
print("--- Huấn luyện mô hình PhoBERT Fine-tuned — 5 lần ---")
!python scripts/train.py --model phobert --num_runs 5

## Bước 5: Đánh giá mô hình trên tập Test

In [18]:
print("=== Đánh giá mô hình BiLSTM ===")
!python scripts/evaluate.py --model bilstm

print("\n=== Đánh giá mô hình BiLSTM + Attention ===")
!python scripts/evaluate.py --model bilstm_attention

# print("\n=== Đánh giá mô hình PhoBERT ===")
# !python scripts/evaluate.py --model phobert

=== Đánh giá mô hình BiLSTM ===
[12:08:21] INFO     Using GPU: Tesla T4 (14.6 GB)                   ]8;id=234053;file:///content/PROJECT-NLP/src/utils/device.py\device.py]8;;\:]8;id=146316;file:///content/PROJECT-NLP/src/utils/device.py#36\36]8;;\
[12:08:21] INFO     Evaluating: bilstm | Checkpoint:              ]8;id=91161;file:///content/PROJECT-NLP/scripts/evaluate.py\evaluate.py]8;;\:]8;id=619176;file:///content/PROJECT-NLP/scripts/evaluate.py#58\58]8;;\
                    models/bilstm/best_model.pt                                 
           INFO     Test set: 10000 samples                       ]8;id=243962;file:///content/PROJECT-NLP/scripts/evaluate.py\evaluate.py]8;;\:]8;id=529903;file:///content/PROJECT-NLP/scripts/evaluate.py#68\68]8;;\
[12:08:21] INFO     Vocabulary loaded from data/vocab/vocab.json    ]8;id=735392;file:///content/PROJECT-NLP/src/data/vocab.py\vocab.py]8;;\:]8;id=571412;file:///content/PROJECT-NLP/src/data/vocab.py#220\220]8

In [20]:
print("Đang tiến hành push lên GitHub...")
!git add .
!git commit -m "Upload training experiments and logs from Google Colab [skip ci]"
!git push origin main
print("=== Push thành công! ===")

Đang tiến hành push lên GitHub...
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
=== Push thành công! ===


## Bước 6: Khởi chạy Web Dashboard trực tiếp từ Colab
Sử dụng công cụ `localtunnel` để ánh xạ port `8501` của Streamlit ra internet công cộng giúp bạn mở giao diện web tương tác.

In [ ]:
# 1. Cài đặt localtunnel
!npm install -g localtunnel

# 2. Chạy Streamlit ngầm trong nền
import subprocess
import time

print("Đang khởi động Streamlit Web Dashboard...")
subprocess.Popen(["streamlit", "run", "app/app.py", "--server.port", "8501"])
time.sleep(5)  # Chờ 5 giây để Streamlit khởi động xong

# 3. Lấy IP Public của máy ảo Colab (Làm mật khẩu đăng nhập của localtunnel)
print("\n=== ĐĂNG NHẬP DASHBOARD ===")
print("Sao chép dãy IP sau đây để dán vào trang web Tunnel (Endpoint IP):")
!curl ipv4.icanhazip.com

# 4. Mở cổng kết nối
print("\nClick vào đường link dưới đây để mở Dashboard tương tác:")
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 5s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹npm notice
npm notice New major version of npm available! 10.8.2 -> 11.17.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.17.0
npm notice To update run: npm install -g npm@11.17.0
npm notice
⠸Đang khởi động Streamlit Web Dashboard...

=== ĐĂNG NHẬP DASHBOARD ===
Sao chép dãy IP sau đây để dán vào trang web Tunnel (Endpoint IP):
8.229.157.128

Click vào đường link dưới đây để mở Dashboard tương tác:
⠙⠹⠸⠼⠴⠦your url is: https://soft-falcons-reply.loca.lt


## Bước 7: Đẩy thẳng các mô hình đã train sang thư mục BTL NLP trên Google Drive

In [ ]:
import os

# Đường dẫn tới thư mục BTL NLP trên Google Drive của bạn
drive_dest_dir = "/content/drive/MyDrive/BTL NLP/models"

# 1. Tạo thư mục models trên Drive nếu chưa tồn tại
!mkdir -p "{drive_dest_dir}"

# 2. Copy toàn bộ các mô hình đã huấn luyện sang Drive
!cp -r /content/PROJECT-NLP/models/* "{drive_dest_dir}/"

print("=== Đã sao lưu toàn bộ mô hình sang thư mục 'BTL NLP' trên Google Drive thành công! ===")
